In [0]:
!pip install yfinance

In [0]:
# Create a sparK session
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("StockPipelie").getOrCreate()
data = [(1, "Gustavo"), (2, "Copilot"), (3, "Databricks")]
df = spark.createDataFrame(data, ["id", "name"])
# df.show()

In [0]:
# Define range date and what tickers to collect
from datetime import datetime, timedelta

# Get yesterday's date
yesterday = datetime.today() - timedelta(days=1)
today_str = datetime.today().strftime("%Y-%m-%d")
yesterday_str = yesterday.strftime("%Y-%m-%d")
print(today, yesterday_str)

# Define tickers used in the process
tickers = ["PETR4.SA", "VALE3.SA", "BBAS3.SA", "ITUB4.SA"]

In [0]:
# Import the most commom Brazilian tickers information
import yfinance as yf

# Example: Download data for Microsoft (MSFT)
df = yf.download(tickers, start=yesterday_str, end=today_str)

# Stack the ticker level into rows
df_long = df.stack().reset_index().rename(columns={"level_2": "Ticker"})

# Convert to Spark DataFrame
spark_df = spark.createDataFrame(df_long)

# Order by Ticker, Date
spark_df = spark_df.orderBy("Date", "Ticker")
spark_df.select("Ticker", "Date", "Close", "High", "Low", "Open", "Volume").show(20)

In [0]:
from pyspark.sql.functions import col, lag
from pyspark.sql.window import Window

# Define window ordered by Date
windowSpec = Window.orderBy("Ticker", "Date")

# Add previous day's Close price
spark_df = spark_df.withColumn("PrevClose", lag("Close").over(windowSpec))

# Calculate daily return
spark_df = spark_df.withColumn(
    "DailyReturn", (col("Close") - col("PrevClose")) / col("PrevClose")
)

spark_df.select("Ticker", "Date", "Close", "PrevClose", "DailyReturn").show(10)